In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import nltk
import string

from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score


nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
file_path = "/content/drive/MyDrive/ambiguity_dataset.xlsx"

df = pd.read_excel(file_path)
df["label"] = df["label"].str.strip().str.lower()
print("Dataset shape:", df.shape)
df.head()

In [ ]:
def preprocess(text):
    tokens = word_tokenize(str(text).lower())
    return " ".join(tokens)

df["processed"] = df["text"].apply(preprocess)

df.head()

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(df["processed"])

y = df["label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

In [ ]:
# --- Replacement for y_pred = model.predict(X_test) ---

# Find the index for the 'ambiguous' class
class_index = list(model.classes_).index("ambiguous")

# Get raw probabilities instead of hard 0/1 labels
y_probs = model.predict_proba(X_test)[:, class_index]

# Set your custom threshold (Tuning for higher Recall)
threshold = 0.42
y_pred_tuned = ["ambiguous" if p >= threshold else "clear" for p in y_probs]

print(f"===== TF-IDF Model (Threshold: {threshold}) =====")
print(classification_report(y_test, y_pred_tuned))



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# BoW Vectorizer
vectorizer_bow = CountVectorizer(ngram_range=(1, 2))
X_bow = vectorizer_bow.fit_transform(df["processed"])

X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

model_bow = LogisticRegression(max_iter=1000)
model_bow.fit(X_train_bow, y_train_bow)

y_pred_bow = model_bow.predict(X_test_bow)

In [ ]:

print("===== TF-IDF Model (Tuned) =====\n")
print(classification_report(y_test, y_pred_tuned)) # Use the tuned labels!

print("\n===== BoW Model =====\n")
print(classification_report(y_test_bow, y_pred_bow))

print("\n===== Accuracy Comparison =====")
print("TF-IDF Accuracy:", round(accuracy_score(y_test, y_pred_tuned), 3))
print("BoW Accuracy:", round(accuracy_score(y_test_bow, y_pred_bow), 3))

In [ ]:
ambiguous_words = [
    "it", "this", "that", "there",
    "some", "many", "few", "near",
    "quickly", "slowly", "something",
    "somewhere", "stuff"
]

pronouns = ["it", "this", "that", "they", "them"]

def rule_based_check(sentence):
    tokens = word_tokenize(sentence.lower())

    found = [w for w in tokens if w in ambiguous_words or w in pronouns]

    if found:
        return "ambiguous", found
    return "clear", []


def highlight_text(sentence, ambiguous_words_found):
    words = sentence.split()

    highlighted = []
    for w in words:
        clean_word = w.lower().strip(string.punctuation)

        if clean_word in ambiguous_words_found:
            highlighted.append(f"[{w.upper()}]")
        else:
            highlighted.append(w)

    return " ".join(highlighted)

In [ ]:
def predict_ambiguity(sentence, threshold=0.35):
    processed = preprocess(sentence)

    # 1. Rule-Based Check (The Safety Net)
    rule_label, words = rule_based_check(sentence)

    # 2. ML Probability Check (Tuned Threshold)
    vec = vectorizer.transform([processed])

    # Dynamic index finding for safety
    class_idx = list(model.classes_).index("ambiguous")
    prob_ambiguous = model.predict_proba(vec)[0][class_idx]

    # Final Logic: Flag if Rules say so OR if ML probability is high enough
    if rule_label == "ambiguous" or prob_ambiguous >= threshold:
        prediction = "AMBIGUOUS"
    else:
        prediction = "CLEAR"

    highlighted_sentence = highlight_text(sentence, words)

    return {
        "sentence": sentence,
        "highlighted": highlighted_sentence,
        "prediction": prediction,
        "confidence": round(prob_ambiguous, 2),
        "ambiguous_words": words
    }

In [ ]:
def predict_ambiguity_bow(sentence, threshold=0.35):
    processed = preprocess(sentence)
    rule_label, words = rule_based_check(sentence)

    vec = vectorizer_bow.transform([processed])
    class_idx = list(model_bow.classes_).index("ambiguous")
    prob_ambiguous = model_bow.predict_proba(vec)[0][class_idx]

    if rule_label == "ambiguous" or prob_ambiguous >= threshold:
        prediction = "AMBIGUOUS"
    else:
        prediction = "CLEAR"

    highlighted_sentence = highlight_text(sentence, words)

    return {
        "sentence": sentence,
        "highlighted": highlighted_sentence,
        "prediction": prediction,
        "confidence": round(prob_ambiguous, 2),
        "ambiguous_words": words
    }

In [ ]:
tests = [
    "Turn off the light",
    "Turn it off",
    "Put it near the box",
    "Email the report to John",
    "Send it quickly",
    "Go there",
    "Open the PDF file in Documents"
]

print("\n===== TF-IDF Predictions =====\n")
for t in tests:
    result = predict_ambiguity(t)
    print(result["highlighted"], "->", result["prediction"], f"(Confidence: {result['confidence']})")

print("\n===== BoW Predictions =====\n")
for t in tests:
    result = predict_ambiguity_bow(t)
    print(result["highlighted"], "->", result["prediction"], f"(Confidence: {result['confidence']})")



In [ ]:
while True:
    text = input("\nEnter instruction (type 'exit' to stop): ")

    if text.lower() == "exit":
        break

    print("\n--- TF-IDF Model ---")
    res1 = predict_ambiguity(text)
    print(res1["highlighted"], "->", res1["prediction"], f"(Confidence: {res1['confidence']})")

    print("\n--- BoW Model ---")
    res2 = predict_ambiguity_bow(text)
    print(res2["highlighted"], "->", res2["prediction"], f"(Confidence: {res2['confidence']})")

    print("-" * 50)